In [1]:
from dataset.cell_motion import CellMotion
try:
    from dataset.cell_motion_rollout import CellMotionRollout
except ImportError:
    from dataset.cell_motion import CellMotionRollout
from model.simulator import Simulator
import torch
from torch_geometric.loader import DataLoader
import torch_geometric.transforms as T
from torch.optim.lr_scheduler import ReduceLROnPlateau
import os
import csv
from datetime import datetime
from typing import Optional

# 数据集路径
dataset_dir = os.path.join(os.getcwd(), 'dataset')

# 训练参数
batch_size = 32  # 批次大小
sequence_length = 20  # 输入的历史时间步数
max_epochs = 1500  # 最大训练轮数

# 预训练模型加载 (新增)
# 如果需要继续训练，请填写checkpoint路径，例如: 'checkpoint/20231027_100000/epoch=100.pth'
load_model_path = None 
# load_model_path = 'checkpoint/20241225_120000/epoch=50.pth' 

# 网络参数
hidden_size=256
message_passing_num = 15  # 消息传递层数
node_input_size = sequence_length * 2 + 1  # 节点输入维度 = 输入的历史时间步数 * 2(x,y坐标) + 1(时间)
edge_input_size = 3  # 边输入维度（笛卡尔坐标差 + 距离）
learning_rate = 1e-03 # 初始学习率

# 学习率调度参数
lr_patience = 50       # 如果loss在50个epoch内没有改善，则调整LR
lr_factor = 0.5        # 学习率调整倍率 (new_lr = lr * factor)
min_lr = 1e-6          # 最小学习率限制

# 训练控制
print_batch = 10  # 每10个批次打印一次损失
save_epoch = 5  # 每5个批次保存一次模型检查点

# ============================================

# 设置计算设备
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# 初始化模拟器模型
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
save_dir = os.path.join(f'checkpoint', timestamp)
if not os.path.exists(save_dir):
    os.makedirs(save_dir)

simulator = Simulator(
    message_passing_num=message_passing_num,
    node_input_size=node_input_size,
    edge_input_size=edge_input_size,
    device=device,
    hidden_size=hidden_size,
    model_dir=save_dir 
)

# 加载预训练权重 (新增)
if load_model_path and os.path.exists(load_model_path):
    print(f'Loading model weights from: {load_model_path}')
    # 注意：根据Simulator的具体实现，可能需要使用 load_state_dict 或自定义的 load 方法
    # 这里假设 Simulator 继承自 nn.Module 或有 load_checkpoint 方法
    try:
        # 尝试使用 Simulator 自带的 load 方法（如果有）
        if hasattr(simulator, 'load_checkpoint'):
             simulator.load_checkpoint(load_model_path)
        else:
            # 标准 PyTorch 加载
            checkpoint = torch.load(load_model_path, map_location=device)
            # 如果checkpoint是字典且包含model_state_dict
            if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
                simulator.load_state_dict(checkpoint['model_state_dict'])
            else:
                simulator.load_state_dict(checkpoint)
        print('Model weights loaded successfully.')
    except Exception as e:
        print(f'Error loading model: {e}')

# 使用Adam优化器
optimizer = torch.optim.Adam(simulator.parameters(), lr=learning_rate)
print('Optimizer initialized')

# 初始化学习率调度器
scheduler = ReduceLROnPlateau(
    optimizer, 
    mode='min', 
    factor=lr_factor, 
    patience=lr_patience, 
    min_lr=min_lr,
    verbose=True
)
print('LR Scheduler initialized')

# 定义图变换流程
transformer = T.Compose([
    T.Cartesian(norm=False),
    T.Distance(norm=False)
])

Using device: cuda
Simulator model initialized with hidden_size=256
Optimizer initialized
LR Scheduler initialized


In [3]:
def train(model: Simulator, dataloader, optimizer, current_epoch, *,
          max_grad_norm: float = 1.0,
          max_grad_value: Optional[float] = None):
    """训练函数"""
    epoch_loss = 0.0
    batch_count = 0
    model.train()
        
    for batch_index, graph in enumerate(dataloader):
        graph = transformer(graph)
        graph = graph.to(device)
        
        node_features = graph.x
        target_positions = graph.y
        current_positions = graph.pos
        
        predicted_output = model.model(graph)

        target_displacement = target_positions - current_positions

        # 训练时更新 normalizer 统计数据
        # 第二个参数传入 model.training (True)
        target_displacement_normalized = model._output_normalizer(
            target_displacement,
            model.training 
        )
        
        mse_loss = torch.mean((predicted_output - target_displacement_normalized) ** 2)
        
        loss = mse_loss 
        
        
        optimizer.zero_grad()
        loss.backward()

        if max_grad_norm is not None and max_grad_norm > 0:
            params_with_grad = [p for group in optimizer.param_groups for p in group['params'] if p.grad is not None]
            if len(params_with_grad) > 0:
                torch.nn.utils.clip_grad_norm_(params_with_grad, max_grad_norm)
        
        optimizer.step()

        epoch_loss += loss.item()
        batch_count += 1

    avg_loss = epoch_loss / batch_count if batch_count > 0 else 0.0
    
    # 定期保存检查点
    if current_epoch % save_epoch == 0 and current_epoch > 0:
        check_point_name = f'epoch={current_epoch}.pth'
        check_point_dir = os.path.join(save_dir, check_point_name)
        model.save_checkpoint(check_point_dir)

    return avg_loss

def validate(model: Simulator, dataloader):
    """验证函数 (新增)"""
    model.eval()
    epoch_loss = 0.0
    batch_count = 0
    
    with torch.no_grad():
        for graph in dataloader:
            graph = transformer(graph)
            graph = graph.to(device)
            
            target_positions = graph.y
            current_positions = graph.pos
            
            # 前向传播
            predicted_output = model.model(graph)
            
            target_displacement = target_positions - current_positions
            
            target_displacement_normalized = model._output_normalizer(
                target_displacement, 
                False 
            )
            
            # 计算MSE Loss
            mse_loss = torch.mean((predicted_output - target_displacement_normalized) ** 2)
            
            epoch_loss += mse_loss.item()
            batch_count += 1
            
    avg_loss = epoch_loss / batch_count if batch_count > 0 else 0.0
    return avg_loss


In [ ]:
if __name__ == '__main__':
    print('=' * 60)
    print('Cell Motion Prediction Training')
    print('=' * 60)
    # ... (参数打印保持不变) ...
    
    # 1. 加载训练集
    dataset_train = CellMotion(
        dataset_dir=dataset_dir,
        split='train',
        max_epochs=max_epochs,
        sequence_length=sequence_length
    )
    train_loader = DataLoader(
        dataset=dataset_train,
        batch_size=batch_size,
        num_workers=0
    )
    
    # 2. 加载验证集 (新增)
    print("Loading Validation Set...")
    dataset_valid = CellMotionRollout(
        dataset_dir=dataset_dir, 
        sequence_length=sequence_length, 
        split='valid'
    )
    valid_loader = DataLoader(
        dataset=dataset_valid, 
        batch_size=batch_size, 
        num_workers=0
    )

    # 训练日志配置
    log_dir = save_dir
    os.makedirs(log_dir, exist_ok=True)
    log_file = os.path.join(log_dir, f'training_log_{timestamp}.csv')
    
    best_epoch_dir = os.path.join(save_dir, 'best_epoch')
    os.makedirs(best_epoch_dir, exist_ok=True)
    
    print(f'\nTraining log will be saved to: {log_file}')
    
    # 更新CSV表头，增加 Valid_Loss
    with open(log_file, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(['Epoch', 'Train_Loss', 'Valid_Loss', 'Timestamp', 'Is_Best', 'Learning_Rate'])
    
    best_valid_loss = float('inf')
    best_epoch_num = 0
    
    print('\nStarting training...\n')
    try:
        for epoch in range(max_epochs):  
            current_epoch = epoch + 1
            
            # 1. 训练
            train_loss = train(simulator, train_loader, optimizer, current_epoch)
            
            # 2. 验证 
            valid_loss = validate(simulator, valid_loader)
            
            # 更新学习率调度器 (使用验证集 loss)
            before_lr = optimizer.param_groups[0]['lr']
            scheduler.step(valid_loss)
            current_lr = optimizer.param_groups[0]['lr']
            
            # 判断是否是最佳模型 (基于验证集 loss)
            is_best = valid_loss < best_valid_loss
            if is_best:
                best_valid_loss = valid_loss
                best_epoch_num = current_epoch
                print(f'\nNew best valid loss: {best_valid_loss:.4e}')
                
                best_checkpoint_path = os.path.join(
                    best_epoch_dir, 
                    f'best_checkpoint.pth'
                )
                simulator.save_checkpoint(best_checkpoint_path)
            
            print(f'Epoch {current_epoch} | Train Loss: {train_loss:.4e} | Valid Loss: {valid_loss:.4e} | LR: {current_lr:.2e}')

            # 写入日志
            with open(log_file, 'a', newline='', encoding='utf-8') as f:
                writer = csv.writer(f)
                current_time = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
                writer.writerow([current_epoch, train_loss, valid_loss, current_time, is_best, current_lr])
            
    except KeyboardInterrupt:
        print('\n\nTraining interrupted by user')
        simulator.save_checkpoint()
        print('Final checkpoint saved')
    
    print('\nTraining completed!')
    print(f'Training log saved to: {log_file}')
    print(f'Best valid loss: {best_valid_loss:.4e} at epoch {best_epoch_num}')

Cell Motion Prediction Training
CellMotion Dataset /data/home/zhaoyj/wansheng/meshgn/meshGraphNets/dataset/train.h5 Initialized (sequence_length=20)
Loading Validation Set...

Training log will be saved to: checkpoint/20260106_213414/training_log_20260106_213414.csv

Starting training...


New best valid loss: 9.1884e-01
Simulator model saved at checkpoint/20260106_213414/best_epoch/best_checkpoint.pth
Epoch 1 | Train Loss: 1.0847e+00 | Valid Loss: 9.1884e-01 | LR: 1.00e-03

New best valid loss: 8.9861e-01
Simulator model saved at checkpoint/20260106_213414/best_epoch/best_checkpoint.pth
Epoch 2 | Train Loss: 1.0403e+00 | Valid Loss: 8.9861e-01 | LR: 1.00e-03
Epoch 3 | Train Loss: 1.0053e+00 | Valid Loss: 9.5582e-01 | LR: 1.00e-03
Epoch 4 | Train Loss: 9.9590e-01 | Valid Loss: 9.4682e-01 | LR: 1.00e-03
Simulator model saved at checkpoint/20260106_213414/epoch=5.pth
Epoch 5 | Train Loss: 9.0403e-01 | Valid Loss: 1.0126e+00 | LR: 1.00e-03
Epoch 6 | Train Loss: 8.5145e-01 | Valid Loss: 9.